## Export to MSTM input format (.ptsa)

`aggregate_ptsa_autogen_hdf5.ipynb` が生成した `aggregates_YYYYMMDD_xx.h5` から、
指定したパラメータの凝集体を1つ読み出し、旧来のCSV形式 `.ptsa` ファイルに書き出す。

**手順**
1. cell-2 を実行して関数を定義する（1回のみ）
2. cell-3 を実行すると利用可能な HDF5 ファイルおよびカタログ CSV の一覧が表示される
3. 対応する `catalog_YYYYMMDD_xx.csv` で目的のパラメータを確認し、`h5_fname` と凝集体パラメータを編集して cell-3 を再実行する

**ファイル命名規則**  
同じ `YYYYMMDD_xx` の HDF5 とカタログ CSV が対応する（例: `aggregates_20260314_00.h5` ↔ `catalog_20260314_00.csv`）。

In [1]:
import h5py
import os
import glob

In [ ]:
def make_h5key(agg_num, k, Df, mean_rp, rel_std_rp, Np):
    """HDF5 group path that uniquely identifies an aggregate by its input parameters.
    Format: '{agg_num}/{k:.3f}/{Df:.2f}/{mean_rp:.4f}/{rel_std_rp:.2f}/{Np:05d}'
    """
    return f"{agg_num}/{k:.3f}/{Df:.2f}/{mean_rp:.4f}/{rel_std_rp:.2f}/{Np:05d}"


def export_to_ptsa(h5_path, agg_num, k, Df, mean_rp, rel_std_rp, Np,
                   output_dir="./generated_agg_files/"):
    """
    Read one aggregate from HDF5 and write a .ptsa file in the original CSV format
    for MSTM input.

    Parameters
    ----------
    h5_path      : str   path to aggregates.h5
    agg_num      : int   aggregate index
    k            : float fractal prefactor
    Df           : float fractal dimension
    mean_rp      : float mean monomer radius [um]
    rel_std_rp   : float relative std of monomer radius [-]
    Np           : int   number of monomers
    output_dir   : str   directory to write the .ptsa file

    Returns
    -------
    fpath : str   path of the written file
    """
    key = make_h5key(agg_num, k, Df, mean_rp, rel_std_rp, Np)

    with h5py.File(h5_path, 'r') as h5f:
        if key not in h5f:
            raise KeyError(
                f"Aggregate not found in HDF5.\n"
                f"  Requested key : {key}\n"
                f"  h5_path       : {h5_path}\n"
                f"Hint: check catalog.csv for available entries."
            )
        grp   = h5f[key]
        xp    = grp["xp"][:]   # shape (Np, 3)  [um]
        rp    = grp["rp"][:]   # shape (Np,)    [um]

    Np_act = xp.shape[0]

    # Filename contains input parameters only (no computed properties)
    ofname = (
        f"agg_num={agg_num:01d}_k={k:.3f}_Df={Df:.2f}"
        f"_meanRp={mean_rp:.3f}um_rstdRp={rel_std_rp:.2f}"
        f"_Np={Np_act:05d}.ptsa"
    )
    fpath = os.path.join(output_dir, ofname)

    os.makedirs(output_dir, exist_ok=True)
    with open(fpath, 'w') as f:
        for ip in range(Np_act):
            f.write("{:13.7e}  {:13.7e}  {:13.7e}  {:13.7e}\n".format(
                xp[ip, 0], xp[ip, 1], xp[ip, 2], rp[ip]))

    print(f"Exported: {fpath}")
    return fpath

In [5]:
out_dir = "./generated_agg_files/"

# ---- 利用可能なファイルを一覧表示 ----
h5_files  = sorted(glob.glob(os.path.join(out_dir, "aggregates_????????_??.h5")))
cat_files = sorted(glob.glob(os.path.join(out_dir, "catalog_????????_??.csv")))
print("Available HDF5 files:")
for f in h5_files  or ["  (none)"]: print(f"  {os.path.basename(f)}")
print("Corresponding catalogs:")
for f in cat_files or ["  (none)"]: print(f"  {os.path.basename(f)}")

Available HDF5 files:
  aggregates_20260314_00.h5
  aggregates_20260314_01.h5
Corresponding catalogs:
  catalog_20260314_00.csv
  catalog_20260314_01.csv


In [6]:


# ---- 出力元ファイルと凝集体パラメータをここで指定 ----
h5_fname   = "aggregates_20260314_01.h5"   # ↑ 上記リストから選択
agg_num    = 0      # aggregate index
k          = 0.95   # fractal prefactor
Df         = 2.35   # fractal dimension
mean_rp    = 0.015  # mean monomer radius [um]
rel_std_rp = 0.10   # relative std of monomer radius [-]
Np         = 350    # number of monomers
# -----------------------------------------------

h5_path = os.path.join(out_dir, h5_fname)

fpath = export_to_ptsa(
    h5_path      = h5_path,
    agg_num      = agg_num,
    k            = k,
    Df           = Df,
    mean_rp      = mean_rp,
    rel_std_rp   = rel_std_rp,
    Np           = Np,
    output_dir   = out_dir,
)

Exported: ./generated_agg_files/agg_num=0_k=0.950_Df=2.35_meanRp=0.015um_rstdRp=0.10_Np=00350_Rve=0.107um_Rg=0.192um_epsagg=0.920.ptsa
